# Stage 2 — Cell Segmentation with CellViT

**Pipeline context:**
```
Stage 1 (local): patchify.py → processed/he/*.png  ←── you upload these
Stage 2 (this notebook): CellViT inference → processed/cellvit/*.json
Stage 3 (local): assign_cells.py → cell type + state masks
```

**What this notebook does:**
1. Installs CellViT and dependencies
2. Downloads the CellViT-256 checkpoint (~400 MB)
3. Reads your `processed/he/` patches from AWS S3
4. Runs CellViT inference (GPU) on each 256×256 H&E patch
5. Post-processes outputs → cell centroids, contours, and type predictions
6. Saves `processed/cellvit/{i}_{j}.json` back to S3

**Before running:**
- Enable GPU: Runtime → Change runtime type → T4 GPU
- Upload your `processed/he/` folder to Google Drive under `he-feature-visualizer/processed/he/`,
  OR upload to S3 and configure the S3 settings below.

## 1 — Configuration
Edit the paths in this cell before running anything else.

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────

STORAGE_BACKEND = 's3'

S3_BUCKET = 'bagherilab-working'
ZIP_KEY   = 'pohao/share_space/tiny_processed.zip'
S3_OUT_PREFIX = 'pohao/he-feature-visualizer/processed/cellvit'
S3_REGION     = 'us-west-2'

# Model variant
# 'CellViT-256'   — 256×256 patches at 40x (default; matches Stage 1 patch size)
# 'CellViT-SAM-H' — higher quality, ~6× slower, needs more VRAM
MODEL_VARIANT = 'CellViT-256'

# Inference settings
BATCH_SIZE    = 32    # patches per forward pass; reduce to 8 if you get OOM
MAGNIFICATION = 40    # 40 for CellViT-256; use 20 for CellViT-256-x20
NUM_WORKERS   = 2

# Skip patches that already have a JSON output (useful to resume interrupted runs)
SKIP_EXISTING = True

# ── DERIVED (do not edit) ─────────────────────────────────────────────────────
import os
COLAB_HE_DIR = '/content/he_patches'     # local copy for fast I/O during inference
CELLVIT_DIR  = '/content/CellViT'
CKPT_DIR     = '/content/checkpoints'

CHECKPOINT_GDRIVE_IDS = {
    'CellViT-256':   '1tVYAapUo1Xt8QgCN22Ne1urbbCZkah8q',
    'CellViT-SAM-H': '1MvRKNzDW2eHbQb5rAgTEp6s2zAXHixRV',
}
CHECKPOINT_FILE = os.path.join(CKPT_DIR, f'{MODEL_VARIANT}.pth')

print(f'Model:            {MODEL_VARIANT}')
print(f'Storage backend:  {STORAGE_BACKEND}')
print(f'Batch size:       {BATCH_SIZE}')

## 2 — Storage Setup (Google Drive or AWS S3)

In [ ]:
import os

import subprocess
subprocess.run(['pip', 'install', '-q', 'boto3'], check=True)
import boto3
import zipfile

try:
    from google.colab import userdata
    AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
    AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
except Exception:
    # Fallback: prompt user
    import getpass
    AWS_ACCESS_KEY_ID     = getpass.getpass('AWS_ACCESS_KEY_ID: ')
    AWS_SECRET_ACCESS_KEY = getpass.getpass('AWS_SECRET_ACCESS_KEY: ')

os.environ['AWS_ACCESS_KEY_ID']     = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
os.environ['AWS_DEFAULT_REGION']    = S3_REGION

s3_client = boto3.client('s3', region_name=S3_REGION)

ZIP_PATH = "/content/tiny_processed.zip"
EXTRACT_DIR = "/content"

s3 = boto3.client("s3")

# download zip
s3.download_file(S3_BUCKET, ZIP_KEY, ZIP_PATH)
print("Downloaded zip")

# unzip
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Unzipped")

HE_DIR  = f"{ZIP_PATH.split(".")[0]}/he"
OUT_DIR = '/content/cellvit_output'
os.makedirs(OUT_DIR, exist_ok=True)


## 3 — Install CellViT
This clones the repo and installs its dependencies. Takes ~2 minutes.

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU in Runtime settings!')

# Clone CellViT
if not os.path.isdir(CELLVIT_DIR):
    !git clone --quiet https://github.com/TIO-IKIM/CellViT.git {CELLVIT_DIR}
    print('Cloned CellViT.')
else:
    print('CellViT already cloned.')

# Add to Python path
if CELLVIT_DIR not in sys.path:
    sys.path.insert(0, CELLVIT_DIR)

# Install requirements (suppress most output)
!pip install -q -r {CELLVIT_DIR}/requirements.txt

# Patch pydantic version conflict (CellViT requires pydantic 1.x, Colab may have 2.x)
!pip install -q 'pydantic==1.10.4'

print('Installation done.')

## 4 — Download Model Checkpoint
CellViT-256 is ~400 MB; CellViT-SAM-H is ~2.5 GB.

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.isfile(CHECKPOINT_FILE):
    size_mb = os.path.getsize(CHECKPOINT_FILE) / 1e6
    print(f'Checkpoint already exists ({size_mb:.0f} MB): {CHECKPOINT_FILE}')
else:
    file_id = CHECKPOINT_GDRIVE_IDS[MODEL_VARIANT]
    print(f'Downloading {MODEL_VARIANT} checkpoint …')
    !pip install -q gdown
    import gdown
    gdown.download(id=file_id, output=CHECKPOINT_FILE, quiet=False)
    size_mb = os.path.getsize(CHECKPOINT_FILE) / 1e6
    print(f'Downloaded: {size_mb:.0f} MB')

# Quick sanity check
import torch
ckpt_meta = torch.load(CHECKPOINT_FILE, map_location='cpu')
print('Checkpoint keys:', list(ckpt_meta.keys())[:8])
run_conf = ckpt_meta.get('run_conf', {})
print('Backbone:', run_conf.get('model', {}).get('backbone', 'not found in run_conf'))

## 5 — Download Patches to Local SSD
Drive/S3 I/O is slow; copying patches locally speeds up inference ~10×.

In [ ]:
import shutil
from tqdm.auto import tqdm

os.makedirs(HE_DIR, exist_ok=True)
local_patches = sorted(p for p in os.listdir(HE_DIR) if p.endswith('.png'))
patches_available = len(local_patches)
print(f'Local patches ready: {len(local_patches)}')

## 6 — Load Model

In [ ]:
import os, sys
import torch
import torch.nn.functional as F
from torchvision import transforms
from collections import OrderedDict

# ---- point this to the folder where you cloned https://github.com/TIO-IKIM/CellViT ----
# It should contain: cell_segmentation/, models/, configs/, ...
CELLVIT_REPO_ROOT = "/content/CellViT"  # <-- change if needed
if CELLVIT_REPO_ROOT not in sys.path:
    sys.path.insert(0, CELLVIT_REPO_ROOT)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("WARNING: No GPU detected. Inference will be extremely slow.")

def strip_prefix_if_present(sd, prefix="module."):
    if not any(k.startswith(prefix) for k in sd.keys()):
        return sd
    new_sd = OrderedDict()
    for k, v in sd.items():
        new_sd[k[len(prefix):] if k.startswith(prefix) else k] = v
    return new_sd

# ── Load checkpoint & read metadata ─────────────────────────────────────────────
checkpoint = torch.load(CHECKPOINT_FILE, map_location="cpu")
arch   = checkpoint.get("arch", "CellViT256")
config = checkpoint.get("config", {}) or {}
print(f"Backbone from checkpoint: {arch}")

# Best-effort: extract config fields (layout varies by checkpoint)
data_conf     = config.get("data", config.get("dataset", {})) if isinstance(config, dict) else {}
model_conf    = config.get("model", {}) if isinstance(config, dict) else {}
training_conf = config.get("training", {}) if isinstance(config, dict) else {}

NUM_NUCLEI_CLASSES = int(data_conf.get("num_nuclei_classes", 6))   # PanNuke-ish default
NUM_TISSUE_CLASSES = int(data_conf.get("num_tissue_classes", 0))   # 0 if not used
drop_rate      = float(training_conf.get("drop_rate", 0.0))
attn_drop_rate = float(training_conf.get("attn_drop_rate", 0.0))
drop_path_rate = float(training_conf.get("drop_path_rate", 0.0))

# encoder weights path (often empty for pure inference checkpoints)
pretrained_encoder = model_conf.get("pretrained_encoder", model_conf.get("model_path", ""))

# ── Select model class based on arch ────────────────────────────────────────────
from models.segmentation.cell_segmentation.cellvit import CellViT256, CellViTSAM

arch_upper = str(arch).upper()
if "SAM" in arch_upper:
    # infer SAM size from arch string, default to SAM-H
    vit_structure = "SAM-H"
    if "SAM-B" in arch_upper: vit_structure = "SAM-B"
    if "SAM-L" in arch_upper: vit_structure = "SAM-L"
    if "SAM-H" in arch_upper: vit_structure = "SAM-H"

    model = CellViTSAM(
        model_path=pretrained_encoder,
        num_nuclei_classes=NUM_NUCLEI_CLASSES,
        num_tissue_classes=NUM_TISSUE_CLASSES,
        vit_structure=vit_structure,
        drop_rate=drop_rate,
    )
else:
    # NOTE: constructor arg is model256_path (not model_path) in this repo
    model = CellViT256(
        model256_path=pretrained_encoder,
        num_nuclei_classes=NUM_NUCLEI_CLASSES,
        num_tissue_classes=NUM_TISSUE_CLASSES,
        drop_rate=drop_rate,
        attn_drop_rate=attn_drop_rate,
        drop_path_rate=drop_path_rate,
    )

# ── Load weights ────────────────────────────────────────────────────────────────
state_dict = checkpoint["model_state_dict"]
state_dict = strip_prefix_if_present(state_dict, "module.")
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Loaded weights. Missing={len(missing)} Unexpected={len(unexpected)}")
if missing[:5]: print("Missing examples:", missing[:5])
if unexpected[:5]: print("Unexpected examples:", unexpected[:5])

model.to(DEVICE).eval()
print(f"Model loaded. Nuclei classes: {NUM_NUCLEI_CLASSES}")

# ── Cell type mapping (PanNuke) ──────────────────────────────────────────────
CELL_TYPE_NAMES = {
    0: "background",
    1: "Neoplastic",
    2: "Inflammatory",
    3: "Connective",
    4: "Dead",
    5: "Epithelial",
}

# ── Image transforms (ImageNet normalisation, matching CellViT training) ──────
inference_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

## 7 — Run Inference
Processes all patches in batches. Each batch takes ~0.5 s on a T4 GPU.  
For 1000 patches this takes roughly 15–20 minutes.

In [ ]:
import json
import cv2
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

os.makedirs(OUT_DIR, exist_ok=True)

# ── Dataset ───────────────────────────────────────────────────────────────────────────
class PatchDataset(Dataset):
    def __init__(self, patch_dir, fnames, transform):
        self.patch_dir = patch_dir
        self.fnames    = fnames
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = Image.open(os.path.join(self.patch_dir, fname)).convert("RGB")
        return self.transform(img), fname


# ── Simple post-processor (cv2 connected components) ─────────────────────────────────
# DetectionCellPostProcessor (HV-map watershed) requires precise HV gradient values
# that are sensitive to model output scale. This approach thresholds the nucleus
# probability map directly and uses cv2 contour detection — robust and interpretable.
#
# At 0.325 µm/px, typical nucleus:
#   diameter ~8–15 µm → 25–46 px → area ~500–1660 px²
# We use a generous range (min=30, max=6000) to catch all nucleus sizes.

MIN_NUCLEUS_AREA = 30    # pixels²  — reject noise/debris
MAX_NUCLEUS_AREA = 6000  # pixels²  — reject large tissue blobs
NUC_THRESHOLD    = 0.5   # foreground probability threshold


def detect_cells(pred_map: np.ndarray) -> list[dict]:
    """
    Args:
        pred_map: (H, W, 4) float32
            ch0 = nuc_prob [0,1]
            ch1 = hv_x
            ch2 = hv_y
            ch3 = type argmax (int)
    Returns:
        List of cell dicts with centroid, contour, bbox, type_cellvit, type_name, type_prob.
    """
    nuc_prob  = pred_map[:, :, 0]
    type_pred = pred_map[:, :, 3].astype(np.int32)

    # Binary mask + morphological opening (remove 1-2 px noise)
    binary = (nuc_prob >= NUC_THRESHOLD).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Connected components
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    cells = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_NUCLEUS_AREA or area > MAX_NUCLEUS_AREA:
            continue

        M = cv2.moments(contour)
        if M['m00'] == 0:
            continue
        cx = M['m10'] / M['m00']
        cy = M['m01'] / M['m00']

        # Cell type: majority vote inside contour mask
        mask = np.zeros(nuc_prob.shape, dtype=np.uint8)
        cv2.drawContours(mask, [contour], -1, 1, thickness=cv2.FILLED)
        type_vals = type_pred[mask == 1]
        cell_type = int(np.bincount(type_vals.clip(0, NUM_NUCLEI_CLASSES - 1)).argmax())

        x, y, w, h = cv2.boundingRect(contour)
        prob_at_centroid = float(nuc_prob[int(cy), int(cx)])

        cells.append({
            'centroid':     [round(cx, 2), round(cy, 2)],
            'contour':      contour.reshape(-1, 2).tolist(),
            'bbox':         [[y, x], [y + h, x + w]],
            'type_cellvit': cell_type,
            'type_name':    CELL_TYPE_NAMES.get(cell_type, 'unknown'),
            'type_prob':    round(prob_at_centroid, 4),
        })

    return cells


# ── Decide which patches to process ───────────────────────────────────────────────────
if SKIP_EXISTING:
    done = {os.path.splitext(f)[0] for f in os.listdir(OUT_DIR) if f.endswith('.json')}
    todo = [f for f in local_patches if os.path.splitext(f)[0] not in done]
    print(f'Skipping {len(done)} already-processed patches.')
else:
    todo = local_patches

print(f'Patches to process: {len(todo)}')

dataset = PatchDataset(HE_DIR, todo, inference_transform)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0,
                     pin_memory=(DEVICE.type == 'cuda'), shuffle=False)

# ── Inference loop ─────────────────────────────────────────────────────────────────────
total_cells = 0
DEBUG_BATCHES = 2

with torch.inference_mode():
    for batch_idx, (imgs, fnames) in enumerate(tqdm(loader, desc='Patches')):
        imgs = imgs.to(DEVICE)

        outputs = model(imgs)

        nuc_prob  = torch.sigmoid(outputs['nuclei_binary_map'])[:, 1, :, :]
        hv_x      = outputs['hv_map'][:, 0, :, :]
        hv_y      = outputs['hv_map'][:, 1, :, :]
        type_pred = torch.argmax(
            torch.softmax(outputs['nuclei_type_map'], dim=1), dim=1
        ).float()

        pred_maps = torch.stack([nuc_prob, hv_x, hv_y, type_pred], dim=-1)
        pred_maps = pred_maps.cpu().numpy().astype(np.float32)

        if batch_idx < DEBUG_BATCHES:
            np0 = pred_maps[0, :, :, 0]
            hx0 = pred_maps[0, :, :, 1]
            print(f"[batch {batch_idx}] nuc_prob: mean={np0.mean():.3f}  frac>0.5={( np0>0.5).mean():.3f} | "
                  f"hv_x: mean={hx0.mean():.3f}  std={hx0.std():.3f}  range=[{hx0.min():.2f},{hx0.max():.2f}]")

        for b, fname in enumerate(fnames):
            patch_id = os.path.splitext(fname)[0]
            cells = detect_cells(pred_maps[b])

            with open(os.path.join(OUT_DIR, f'{patch_id}.json'), 'w') as f:
                json.dump({'patch': patch_id, 'cells': cells}, f)

            total_cells += len(cells)

print(f'\nDone. {len(todo)} patches, {total_cells:,} cells total.')
print(f'Avg cells/patch: {total_cells / max(len(todo), 1):.1f}')


## 8 — Verify Output
Spot-check one patch result and visualise cells on the H&E.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

TYPE_COLORS = {
    0: '#808080',   # background / other
    1: '#DC3232',   # Neoplastic   — red
    2: '#3264DC',   # Inflammatory — blue
    3: '#32B432',   # Connective   — green
    4: '#8B008B',   # Dead         — purple
    5: '#FF8C00',   # Epithelial   — orange
}

# Pick first available JSON
json_files = sorted(f for f in os.listdir(OUT_DIR) if f.endswith('.json'))
assert json_files, 'No JSON files found in output directory.'

sample_id = os.path.splitext(json_files[0])[0]
with open(os.path.join(OUT_DIR, json_files[0])) as f:
    result = json.load(f)

he_path = os.path.join(HE_DIR, f'{sample_id}.png')
cells   = result['cells']

# ── Summary ───────────────────────────────────────────────────────────────────
type_counts  = Counter(c['type_name']    for c in cells)
print(f"Patch {sample_id}: {len(cells)} cells")
print("  Cell type breakdown:")
for name, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    bar = '█' * min(count, 40)
    print(f"    {name:<15} {count:>4}  {bar}")

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(5, 3))
fig.patch.set_facecolor('black')
he_img = np.array(Image.open(he_path).convert('RGB'))

for ax in axes:
    ax.set_facecolor('black')
    ax.axis('off')

axes[0].imshow(he_img)
axes[0].set_title(f'H&E — patch {sample_id}', color='white')
axes[1].imshow(he_img)

CONTOUR_KEYS = ("contour", "boundary", "polygon", "coords", "outline")

n_drawn = 0
for cell in cells:
    # pick a contour field if available
    contour = None
    for k in CONTOUR_KEYS:
        if k in cell and cell[k] is not None:
            contour = cell[k]
            break
    if contour is None:
        continue
    if isinstance(contour, dict) and "x" in contour and "y" in contour:
        pts = np.column_stack([contour["x"], contour["y"]])
    else:
        pts = np.asarray(contour)

    if pts.ndim != 2 or pts.shape[1] != 2 or len(pts) < 3:
        continue

    color = TYPE_COLORS.get(cell.get("type_cellvit", 0), "#808080")

    # Draw polygon contour
    poly = mpatches.Polygon(
        pts,
        closed=True,
        fill=False,
        edgecolor=color,
        linewidth=2,
        alpha=0.9,
    )
    axes[1].add_patch(poly)
    n_drawn += 1

legend_handles = [
    mpatches.Patch(color=TYPE_COLORS[t], label=CELL_TYPE_NAMES[t])
    for t in range(1, 6)
]
axes[1].legend(handles=legend_handles, loc='upper right', fontsize=7,
               facecolor='#222222', labelcolor='white', edgecolor='gray')
axes[1].set_title(f'CellViT types — {len(cells)} cells', color='white')

plt.tight_layout()
plt.show()


## 9 — Upload Results to S3
Syncs completed JSON files back to your chosen storage backend.

In [ ]:
import os
import zipfile
from tqdm import tqdm

# Collect outputs
json_files_out = sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".json"))
print(f"JSON files produced: {len(json_files_out)}")
assert json_files_out, "No JSON files found to upload."

# ---- Zip outputs locally ----
ZIP_NAME = "cellvit_outputs.zip"
ZIP_PATH = os.path.join("/content", ZIP_NAME)  # Colab local disk

# (optional) overwrite if exists
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

print(f"Zipping {len(json_files_out)} JSON files -> {ZIP_PATH}")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fname in tqdm(json_files_out, desc="Zipping"):
        full_path = os.path.join(OUT_DIR, fname)
        # store files inside cellvit/ folder inside zip
        zf.write(full_path, arcname=f"cellvit/{fname}")

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024**2)
print(f"Zip created: {ZIP_PATH} ({zip_size_mb:.2f} MB)")

# ---- Upload the zip to S3 ----
zip_s3_key = f"{S3_OUT_PREFIX}/{ZIP_NAME}"
print(f"Uploading zip to s3://{S3_BUCKET}/{zip_s3_key} ...")
s3_client.upload_file(ZIP_PATH, S3_BUCKET, zip_s3_key)
print("Upload complete.")

print("\nDownload locally with:")
print(f"  aws s3 cp s3://{S3_BUCKET}/{zip_s3_key} .")
print(f"  unzip -q {ZIP_NAME} -d processed/cellvit/")


**After downloading results:**  
Copy `processed/cellvit/` to your local `he-feature-visualizer/processed/cellvit/` and run Stage 3:
```bash
python assign_cells.py \
  --cellvit-dir processed/cellvit/ \
  --features-csv data/CRC02.csv \
  --out processed/index.json
  --index 
```